# UAVIDS-2025 — benchmark local de inferência

Este notebook apresenta a primeira medição de sistemas dos artefatos congelados. A fronteira começa imediatamente antes de `predict_proba` e termina no retorno das probabilidades. A leitura do CSV, a rede e a serialização não participam do tempo.

## Protocolo e cautela

O procedimento está em [`local_benchmark_v1.md`](../../protocol/local_benchmark_v1.md). Foram feitas 200 chamadas de aquecimento e cinco repetições, totalizando 5.000 chamadas individuais por modelo. Os dados de duração brutos foram preservados. Este é um piloto em Windows compartilhado, sem restrição de CPU/RAM.

In [1]:
from pathlib import Path
import json
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "research":
    project_root = project_root.parents[1]
elif project_root.name == "notebooks":
    project_root = project_root.parent
benchmark_dir = project_root / "benchmarks" / "local_inference_v1"
report_dir = project_root / "reports" / "local_inference_v1"
manifest = json.loads((benchmark_dir / "manifest.json").read_text("utf-8"))
print(json.dumps(manifest, indent=2, ensure_ascii=False))

{
  "benchmark_id": "local_inference_v1",
  "status": "pilot_local_in_process",
  "config_sha256": "f7dcb2c81bbc73a6f55f5605e4c4351a42cf110021671e37ca8c713703ecfe5c",
  "completed_models": 2,
  "expected_models": 2,
  "complete": true,
  "models": [
    "xgboost",
    "random_forest"
  ],
  "model_summaries": [
    "xgboost__summary.json",
    "random_forest__summary.json"
  ]
}


## Latência individual, carregamento e memória residente

In [2]:
individual = pd.read_csv(report_dir / "individual_latency.csv")
print(individual.round(3).to_string(index=False))

        model  model_mib  load_ms  rss_load_delta_mib  count   mean_us   std_us   p50_us    p95_us    p99_us  max_us
      xgboost      2.756   56.196              29.273   5000   244.530   36.051   236.30   306.620   346.912  1049.9
random_forest    192.704  159.085             194.828   5000 32727.553 3620.332 33887.65 34808.515 35725.746 65630.5


XGBoost apresentou P50 **236.30 µs** e P99 **346.91 µs**. RF apresentou P50 **33887.65 µs** e P99 **35725.75 µs**. A diferença de P50 foi de 143.4 vezes neste host.

## Lotes e custo amortizado

In [3]:
batches = pd.read_csv(report_dir / "batch_latency.csv")
print(batches.round(3).to_string(index=False))

        model  count   mean_us   std_us   p50_us    p95_us    p99_us  max_us  batch_size  amortized_p50_us_per_row  median_rows_per_second
      xgboost    150   477.239  165.379   434.45   783.585   921.351  1035.5           1                   434.450                2301.823
      xgboost    150  1102.003  193.112  1081.45  1453.065  1530.381  1685.4          32                    33.795               29589.908
      xgboost    150  1849.392  260.724  1858.05  2273.645  2450.883  2524.1         256                     7.258              137778.883
      xgboost    150  6298.341  695.467  6453.95  6938.160  7563.568  8786.2        1024                     6.303              158662.542
random_forest    150 33230.265 3072.259 34081.30 35149.760 35648.048 35720.7           1                 34081.300                  29.342
random_forest    150 34290.913 2533.381 33988.45 34743.890 40745.860 62300.7          32                  1062.139                 941.496
random_forest    150 35261.

![Amortização por lote](../../reports/local_inference_v1/batch_amortization.png)

## Conclusão operacional limitada

XGBoost é o candidato preferencial para medir requisição em container porque manteve a melhor qualidade média, usa artefato muito menor e foi substancialmente mais rápido neste piloto. RF permanece como comparador. A próxima bancada deve usar exatamente os hashes congelados, medir cliente–servidor e registrar recursos/limites efetivamente aplicados. Não se infere desempenho em drone a partir deste host.